In [5]:
# -----------------------
#  CELL 1: CONFIG SECTION
# -----------------------

# Where to find matrices and where to save output
CONFIG = {
    "matrices_dir": "../postproc_intermediate/matrices",
    "output_tex": "../postproc_outputs/small_table.tex",

    # Paths for LaTeX components
    "preamble_path": "../postproc_inputs/latex_preamble_smalltable.tex",
    "fragment_template_path": "../postproc_inputs/latex_smalltable_fragment.tex.j2",
    "closing_path": "../postproc_inputs/latex_closing_smalltable.tex",
}

# Models, references, prompts, metrics to include
MODELS = ["llama", "gpt"]
REFERENCES = ["gold", "silver"]

# These must match the column names in *_overall_all_pretty.csv
PROMPTS = {
    "zs": "zeroShot",
    "zscot": "zeroShotCoT",
}

# Metrics available as pretty CSVs
METRICS = ["qwk", "pearson", "icc3", "mae", "rmse", "deltaT", "w"]


In [6]:
# ---------------------------
#  CELL 2: HELPER FUNCTIONS
# ---------------------------

import pandas as pd
from pathlib import Path
from jinja2 import Environment, FileSystemLoader

def load_matrix_value(model, reference, metric, prompt_col):
    """
    Load the value for a given (model, reference, metric, prompt).
    Restrict to manual extraction, subgroup overall/all.
    
    Parameters
    ----------
    model : str
        "llama" or "gpt"
    reference : str
        "gold" or "silver"
    metric : str
        one of ["qwk", "pearson", "mae", "rmse", "deltaT", "w"]
    prompt_col : str
        must match a column in the CSV (e.g., "zeroShot", "zeroShotCoT")
    
    Returns
    -------
    str
        LaTeX-ready value (mean ± sd for metrics, or Δ,t / W with stars).
    """
    fname = f"{metric}_overall_all_pretty.csv"
    fpath = Path(CONFIG["matrices_dir"]) / model / reference / "aggregate" / fname
    
    df = pd.read_csv(fpath)
    row = df[df["extract"] == "manual"]
    if row.empty:
        return "–"
    return str(row.iloc[0][prompt_col])


def boldface_preferable(val1, val2, metric):
    """
    Compare two LaTeX strings (val1=LLaMA, val2=GPT) and boldface the preferable one.
    
    Rules:
      - For QWK, Pearson r, and ICC3: higher mean is better.
      - For MAE and RMSE: lower mean is better.
      - If means are equal (within tolerance), prefer the smaller stdev.
      - For deltaT and W: leave untouched.
    """
    try:
        # Expect strings like "3.39 ± 0.37"
        parts1 = val1.replace("$", "").split("±")
        parts2 = val2.replace("$", "").split("±")
        
        mean1 = float(parts1[0].strip().split()[0])
        mean2 = float(parts2[0].strip().split()[0])
        
        sd1 = float(parts1[1].strip()) if len(parts1) > 1 else None
        sd2 = float(parts2[1].strip()) if len(parts2) > 1 else None
    except Exception:
        return val1, val2
    
    tol = 1e-6  # tolerance for treating means as equal
    
    if metric in ["qwk", "pearson", "icc3"]:
        if abs(mean1 - mean2) > tol:
            if mean1 > mean2:
                return f"\\textbf{{{val1}}}", val2
            elif mean2 > mean1:
                return val1, f"\\textbf{{{val2}}}"
        else:
            # Tie on mean → compare standard deviation
            if sd1 is not None and sd2 is not None:
                if sd1 < sd2:
                    return f"\\textbf{{{val1}}}", val2
                elif sd2 < sd1:
                    return val1, f"\\textbf{{{val2}}}"
    
    elif metric in ["mae", "rmse"]:
        if abs(mean1 - mean2) > tol:
            if mean1 < mean2:
                return f"\\textbf{{{val1}}}", val2
            elif mean2 < mean1:
                return val1, f"\\textbf{{{val2}}}"
        else:
            # Tie on mean → compare standard deviation
            if sd1 is not None and sd2 is not None:
                if sd1 < sd2:
                    return f"\\textbf{{{val1}}}", val2
                elif sd2 < sd1:
                    return val1, f"\\textbf{{{val2}}}"
    
    return val1, val2


In [7]:
# -------------------------
#  CELL 3: BUILD CONTEXT
# -------------------------

# Initialize empty nested dict for all metrics / references / prompts
context = {m: {ref: {p: {} for p in PROMPTS} for ref in REFERENCES} for m in METRICS}

for metric in METRICS:
    for reference in REFERENCES:
        for prompt_key, prompt_col in PROMPTS.items():
            # Load values for LLaMA and GPT
            llama_val = load_matrix_value("llama", reference, metric, prompt_col)
            gpt_val   = load_matrix_value("gpt", reference, metric, prompt_col)
            
            # Apply boldfacing for the performance metrics
            if metric in ["qwk", "pearson", "icc3", "mae", "rmse"]:
                llama_val, gpt_val = boldface_preferable(llama_val, gpt_val, metric)
            
            # Store into context structure
            context[metric][reference][prompt_key]["llama"] = llama_val
            context[metric][reference][prompt_key]["gpt"]   = gpt_val

# Quick sanity check: print one example row to confirm structure
import pprint
pprint.pprint(context["mae"]["gold"]["zs"])


{'gpt': '\\textbf{2.50 ± 0.07}', 'llama': '3.13 ± 0.07'}


In [8]:
# -------------------------
#  CELL 4: RENDER LATEX
# -------------------------

# Set up Jinja2 environment
env = Environment(loader=FileSystemLoader(Path(CONFIG["fragment_template_path"]).parent))
template = env.get_template(Path(CONFIG["fragment_template_path"]).name)

output_path = Path(CONFIG["output_tex"])

with open(output_path, "w") as f_out:
    # Write LaTeX preamble
    f_out.write(Path(CONFIG["preamble_path"]).read_text())
    
    # Render the table fragment with context
    f_out.write(template.render(
        qwk=context["qwk"],
        r=context["pearson"],
        icc3=context["icc3"],
        mae=context["mae"],
        rmse=context["rmse"],
        deltat=context["deltaT"],
        w=context["w"]
    ))
    
    # Write LaTeX closing
    f_out.write(Path(CONFIG["closing_path"]).read_text())

print(f"LaTeX small table written to {output_path}")


LaTeX small table written to ..\postproc_outputs\small_table.tex
